# Attribute extraction pipeline

## Expert-based extraction 

In [3]:
from google import genai
from google.genai import types
import os
client = genai.Client(api_key=os.environ["GENAI_API_KEY"])

In [1]:
dataset_name = "../data/categorization/DKK-other-final.csv" 

In [8]:
import json 
def get_json(string):
    stripped = ""
    braces = 0
    for s in string:
        if s == "{":
            braces += 1
        if braces >= 1:
            stripped += s
        if s == "}":
            braces -= 1
    return json.loads(stripped, strict=False)


In [3]:
import pandas as pd
import csv
import sys
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    dataset_name,
    sep=",",
    quoting=1,  # csv.QUOTE_ALL
    quotechar='"',
    escapechar="\\",
    engine="python"  # Required when dealing with multiline quoted fields
)

# Check the first few rows
marenky = df[df["judgement_model_short"] == "A"]
pod_vlivem = df[df["judgement_model_short"] == "B"]
s_nasledkem = df[df["judgement_model_short"] == "C"]
jina = df[df["judgement_model_short"] == "D"]
nedopravni = df[df["judgement_model_short"] == "E"]

In [13]:
print(f"Violation of a driving ban: {len(marenky)}")
print(f"Drug-impaired driving: {len(pod_vlivem)}")
print(f"Traffic-related injury: {len(s_nasledkem)}")
print(f"Other traffic offenses: {len(jina)}")
print(f"Non-traffic offenses: {len(nedopravni)}")


Violation of a driving ban: 369
Drug-impaired driving: 1775
Traffic-related injury: 983
Other traffic offenses: 103
Non-traffic offenses: 6740


## Violation of a driving ban

In [11]:

MARENKY_STRUCTURED = {}
MARENKY_EXTRA = {}


In [12]:
def extract(marenka):
    print(marenka["judgement_slt_id"])
    print(marenka["judgement_factual_sentence"])
    chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(system_instruction="""You extract values for attributes from text. The texts are the factual statements in court verdicts. Return only information from the text, never make stuff up. If the value for attribute is not provided, return value 'n/a'. The date format id DD.MM.YYYY.  The time format is HH:MM. Decimal number are rounded to two digits after the decimal point: #.##. Format your output as JSON. The values mut be in the original language - Slovak. Extract the follwing attributes:
            revokingAuthority [Authority that revoked the driving license]  
            authorityLocation [Location of the authority that revoked the license]  
            revocationDateLegalEffect [Date when the license revocation became legally binding]  
            decisionDate [Date of the decision]  
            revocationDuration [For how long the license was revoked]  
            dateOfOffense [Date when the offense occurred]  
            timeOfOffense [Time when the offense occurred]  
            placeOfOffense [A name of the city/town where the offense occurred]
            placeDetails [Details about where the offense occurred]
            vehicleModel [Vehicle brand and model involved in the offense, e.g Škoda Octavia, or "anonymized" if only capital letters are provided]
            vehicleType [Type of vehicle involved in the offense]
            checkedBy [Who conducted the inspection or check]  
            typeOfDrivingBan [Type of driving ban / which vehicles are affected]  
            typeOfRoad [Type of road where the offense occurred]  
            reasonForRevocation [Reason for revoking the driving license]  
            additionalViolations [Additional traffic law violations] 

            The placeOfOffense is coded as follows:
            [name of the town/city in Slovak]
            outside town/city limits (if offence occurs outside of any municipality)
            anonymized (e.g. v obci A.B.)
            n/a (if the information in not provided)

            The vehicleType is coded as follows:
            car
            van
            truck
            bus
            motorcycle
            other

            The typeOfRoad is coded as follows:
            urban
            motorway
            class_I
            class_II
            other
            n/a (if the information in not provided)
            """),
       )
    
    response = chat.send_message(marenka["judgement_factual_sentence"])
    MARENKY_STRUCTURED[marenka["judgement_slt_id"]] = get_json(response.text)
    
    print(f"ATTRIBUTES:\n{response.text}")
    
    #response = chat.send_message("What other information was there in the statenent but is not covered by the attributes? Write is in a structured way as JSON (without high level attribute, e.g. OtherInformatio). Only output information from the original text.  The keys must be in English, the values mut be in the original language - Slovak.")
    #MARENKY_EXTRA[marenka["judgement_slt_id"]] = get_json(response.text)
 
    response = chat.send_message("What other information was there in the statenent but is not covered by the attributes? Write one sentence containing the extra information. Output a json with one atttribute: 'note' wherethe value is the sentence with extra information. Only output information from the original text. The sentence mut be in the original language - Slovak.")
    MARENKY_STRUCTURED[marenka["judgement_slt_id"]]["note"] = get_json(response.text)["note"]
    MARENKY_STRUCTURED[marenka["judgement_slt_id"]]["sentence"] = marenka["judgement_factual_sentence"]
    
    
    print(f"EXTRA:\n{response.text}")

In [84]:
for i in range(len(marenky)): 
    marenka = marenky.iloc[i]
    extract(marenka)
    


2154941
, ž e  dňa 30.07.2017 v čase o 8:23 hod. viedol na ceste II. triedy č. 425 na hraničnom prechode Brodské - Lanžhot osobné motorové vozidlo zn. C. N., EČV: XBXXXXX, kde bol zastavený a kontrolovaný hliadkou OO PZ Brodské, pričom kontrolou bolo zistené, že viedol motorové vozidlo i napriek tomu, že má rozsudkom Okresného súdu Senica sp.zn. 1T/96/2014 zo dňa 27.10.2014, právoplatný 27.10.2014 uložený zákaz činnosti viesť motorové vozidlá na dobu 12 (dvanásť) mesiacov, ktorý mu začal plynúť až po prepustení z ÚVTOS na slobodu dňa 27.12.2016,
ATTRIBUTES:
```json
{
  "revokingAuthority": "Okresný súd Senica",
  "authorityLocation": "Senica",
  "revocationDateLegalEffect": "27.10.2014",
  "decisionDate": "27.10.2014",
  "revocationDuration": "12 mesiacov",
  "dateOfOffense": "30.07.2017",
  "timeOfOffense": "08:23",
  "placeOfOffense": "Brodské",
  "placeDetails": "na ceste II. triedy č. 425 na hraničnom prechode Brodské - Lanžhot",
  "vehicleModel": "anonymized",
  "vehicleType": "ca

In [86]:
# augment with city size
for key in MARENKY_STRUCTURED:
    placeOfOffense = MARENKY_STRUCTURED[key]["placeOfOffense"]
    MARENKY_STRUCTURED[key]["citySize"] = cities.get(placeOfOffense, "n/a")    


In [87]:
df = pd.DataFrame.from_dict(MARENKY_STRUCTURED, orient="index").reset_index()
df.to_csv("marenky_attributes_updated_with_city_size.csv", index=False)
df


,index,revokingAuthority,authorityLocation,revocationDateLegalEffect,decisionDate,revocationDuration,dateOfOffense,timeOfOffense,placeOfOffense,placeDetails,vehicleModel,vehicleType,checkedBy,typeOfDrivingBan,typeOfRoad,reasonForRevocation,additionalViolations,note,sentence,citySize
0,2154941,Okresný súd Senica,Senica,27.10.2014,27.10.2014,12 mesiacov,30.07.2017,08:23,Brodské,na ceste II. triedy č. 425 na hraničnom precho...,anonymized,car,hliadkou OO PZ Brodské,zákaz činnosti viesť motorové vozidlá,class_II,"viedol motorové vozidlo i napriek tomu, že má ...",n/a,Zákaz činnosti viesť motorové vozidlá mu začal...,", ž e dňa 30.07.2017 v čase o 8:23 hod. viedo...",n/a
1,1636460,Okresného súdu Bratislava II,Bratislava II,25.10.2017,25.10.2017,36 mesiacov,22.01.2018,23:10,Stupava,smere zo Zdravotníckej ulice na Hlavnú ulicu,anonymized,car,n/a,akéhokoľvek druhu,urban,"Viedol motorové vozidlo napriek tomu, že mu bo...",n/a,Trestný rozkaz mal číslo 0T/357/2017 a vozidlo...,dňa 22.01.2018 v čase okolo 23.10 h v Stupave ...,12108
2,2162446,Okresného dopravného inšpektorátu Michalovce,Michalovce,28.09.2017,06.09.2017,12 mesiacov,30.01.2018,14:30,Žbince,v smere od základnej školy smerom k rómskej osade,anonymized,car,policajnou hliadkou,viesť motorové vozidlo,class_III,priestupok proti bezpečnosti a plynulosti cest...,nesvietili stretávacie svetlá,Rozhodnutie Okresného dopravného inšpektorátu ...,"aj napriek tomu, že rozhodnutím Okresného dopr...",n/a
3,1638701,Okresný súd Malacky,Malacky,19.05.2017,19.05.2017,15 mesiacov,25.01.2018,02:50,Stupava,Železničná ulica,anonymized,car,n/a,viesť motorové vozidlá akéhokoľvek druhu,urban,Viedol motorové vozidlo napriek právoplatnému ...,n/a,Vozidlo malo e.č.: MA-XXX DB a zákaz činnosti ...,dňa 25.01.2018 v čase okolo 02.50 h v Stupave ...,12108
4,2221001,Okresný súd Námestovo,Námestovo,12.03.2016,25.02.2016,40 mesiacov,19.01.2018,13:33,Trstená,na ulici Oravickej pri rodinnom dome číslo XXX/XX,KUBOTA,other,hliadka Obvodného oddelenia PZ Trstená,všetky motorové vozidlá,urban,Šoférovanie počas zákazu činnosti,n/a,"Ďalšie informácie v texte, ktoré neboli pokryt...","i napriek tomu, že trestným rozkazom Okresného...",7360
5,1641467,Okresný dopravný inšpektorát Okresného riadite...,Komárno,27.12.2017,07.12.2017,12 mesiacov,01.02.2018,17:32,Komárno,Eötvösova ulica,Ford Focus,car,Hliadka Obvodného oddelenia Policajného zboru ...,viesť motorové vozidlá,urban,Vedenie motorového vozidla napriek uloženému z...,Nepredloženie vodičského preukazu pri kontrole,"Evidenčné číslo vozidla bolo G.XXXW., hliadka ...",", že dňa 1. februára 2018 v čase o 17.32 hodin...",33751
6,2162837,Okresného súdu v Bratislave 1,Bratislave 1,26.01.2014,10.11.2013,3 rokov,22.02.2015,14:10,anonymized,"po ceste č. I/XX v smere od P. A. na U., na C. Z.",anonymized,car,"hliadkou OR PZ, ODI Lučenec",viesť motorové vozidlá akéhokoľvek druhu,class_I,marenia výkonu úradného rozhodnutia podľa § 34...,prekročil dovolenú rýchlosť jazdy,Páchateľovi bol uložený peňažný trest vo výmer...,", ž e dňa 22.02.2015 v čase o 14.10 hod. po c...",n/a
7,2009959,ODI Bratislava V,Bratislava V,30.08.2016,01.08.2016,24 mesiacov,06.02.2018,10:08,Malacky,ul. Duklianskych hrdinov,anonymized combi,car,hliadkou ODI Malacky,viesť motorové vozidlá,urban,n/a,n/a,Okrem zákazu činnosti uloženého rozhodnutím OD...,", ž e\ndňa 06.02.2018 v čase o 10.08 h. viedol...",17467
8,2169234,Okresný dopravný inšpektorát,Prešov,09.05.2016,09.05.2016,60 mesiacov,29.08.2016,12:55,Čierne nad Topľou,smerom od mesta Prešov k mestu Vranov nad Topľou,anonymized,car,ODI Vranov nad Topľou,viesť motorové vozidlá všetkých druhov,class_I,riadenie vozidla napriek uloženému zákazu činn...,n/a,Vozidlo malo evidenčné číslo D. Č. K.-XXXFM a ...,", ž e dňa 29.08.2016, v čase o 12:55 hod., vi...",n/a
9,2351172,Okresný dopravný inšpektorát Dunajská Streda,Dunajská Streda,31.10.2017,20.09.2017,24 mesiacov,08.02.2018,15:00,anonymized,"H. L. P., okres Dunajská Streda",Fiat Brava,car,hliadka Obvodného oddelen

## Drug-impaired driving

In [7]:
POD_VLIVEM_STRUCTURED = {}

In [8]:
def extract_drugs(pod_vlivem):
    print(pod_vlivem["judgement_slt_id"])
    print(pod_vlivem["judgement_factual_sentence"])
    chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(system_instruction="""You extract values for attributes from text. The texts are the factual statements in court verdicts. Return only information from the text, never make stuff up. If the value for attribute is not provided, return value 'n/a'. The date format id DD.MM.YYYY.  The time format is HH:MM. Decimal number are rounded to two digits after the decimal point: #.##. Format your output as JSON. The values mut be in the original language - Slovak. Extract the follwing attributes:
            substanceUsed [Drug or substance detected]
            substanceUsedType [Type of the substance]
            amountUsed [The amount of drug or substance detected]
            measuringMethod [Type of measurement taken]
            measuringTime [Time of measurement]
            measuringDevice [Measuring device used for testing]
            refusedTesting [True if the offender refused to take a blood test, False otherwise]
            dateOfOffense [Date when the offense occurred]
            timeOfOffense [Time when the offense occurred]
            placeOfOffense [A name of the city/town where the offense occurred]
            placeDetails [Details about where the offense occurred]
            vehicleModel [Vehicle brand and model involved in the offense, e.g Škoda Octavia, or "anonymized" if only capital letters are provided]
            vehicleType [Type of vehicle involved in the offense]
            checkedBy [Who conducted the inspection or check]
            typeOfRoad [Type of road where the offense occurred]
            reasonForCheck [Reason why the inspection was carried out]
            reasonForCheckCategory [Reason why the inspection was carried out]
            licenseStatus [Validity status of the driver's linense]
            lawBroken [A reference to a law that was broken by the offense]
            
            The substanceUsed is coded as follows:
            alcohol
            cannabis
            methamphetamine
            amphetamine       
            cocaine
            mdma
            hashish    
            morphine
            barbiturate 
            benzodiazepine
            methadon 
            heroin
            other

            where ethanol refers to alcohol as welll

            The amountUsed is composed of a number and a unit of measurement, e.g. 0.75 mg/l, 50 ng/ml, 0.70 g/kg

            The substanceUsedType is coded as follows:
            stimulants
            sedatives
            hallucinogens

            where sedatives refer to sedatives or depressants.

            The measuringMethod is coded as follows:
            breath
            blood
            urine
            other
            n/a

            If there are multiple measurements of the same substance, list all of them separated by semicolon, e.g. amountUsed: 0.75 mg/l; 0.5 mg/l, measuringMethod: blood; blood, measuringTime: 22:34; 22:42. When only one type of substance was used, this only applies to attributes amountUsed, measuringMethod, measuringTime whereas substanceUsed has only one value.  However, if there is a use of diffeent substances, then the attributes substanceUsed and substanceUsedType also have multiple values separated by semicolon, e.g. substanceUsed: alcohol; cannabis, amountUsed: 45 ng/l; 22 ng/l. Do not report alcohol values in ‰.

            The placeOfOffense is coded as follows:
            [name of the town/city in Slovak]
            outside town/city limits (if offence occurs outside of any municipality)
            anonymized (e.g. v obci A.B.)
            n/a (if the information in not provided)

            The vehicleType is coded as follows:
            car
            van
            truck
            bus
            motorcycle
            other
            n/a

            The typeOfRoad is coded as follows:
            urban
            motorway
            class_I
            class_II
            other
            n/a (if the information in not provided)

            reasonForCheckCategory is coded as follows:
            traffic_accident
            speeding
            traffic_violation
            suspected_intoxication
            traffic_check
            other
            n/a

            The licenseStatus is coded as follows:
            valid (valid license plate)
            banned (driving despite a ban was imposed or license was taken away)
            none (license never issued, driving without driver's license)
            n/a (license validity not mentioned)
  
"""),
       )
    
    response = chat.send_message(pod_vlivem["judgement_factual_sentence"])
    POD_VLIVEM_STRUCTURED[pod_vlivem["judgement_slt_id"]] = get_json(response.text)
    
    print(f"ATTRIBUTES:\n{response.text}")
      
    response = chat.send_message("What other information was there in the statenent but is not covered by the attributes? Write one sentence containing the extra information. Output a json with one atttribute: 'note' wherethe value is the sentence with extra information. Only output information from the original text. The sentence mut be in the original language - Slovak.")
    POD_VLIVEM_STRUCTURED[pod_vlivem["judgement_slt_id"]]["note"] = get_json(response.text)["note"]
    POD_VLIVEM_STRUCTURED[pod_vlivem["judgement_slt_id"]]["sentence"] = pod_vlivem["judgement_factual_sentence"]
   
    
    print(f"EXTRA:\n{response.text}")

In [ ]:
import time
for i in range(len(pod_vlivem)):     
    pod_vlivem_i = pod_vlivem.iloc[i]
    print(i)
    print(pod_vlivem_i)
    try:
        extract_drugs(pod_vlivem_i)
    except Exception as e:
        time.sleep(30)
        print("Another attempt...")
        extract_drugs(pod_vlivem_i)

## Enrichment with external data

In [15]:
cities = {
  "Bratislava": 437725,
  "Košice": 238593,
  "Prešov": 88464,
  "Žilina": 80727,
  "Banská Bystrica": 78084,
  "Nitra": 76533,
  "Trnava": 65033,
  "Trenčín": 55383,
  "Martin": 54168,
  "Poprad": 51235,
  "Prievidza": 45634,
  "Zvolen": 42167,
  "Považská Bystrica": 39271,
  "Michalovce": 38776,
  "Nové Zámky": 37512,
  "Spišská Nová Ves": 37007,
  "Komárno": 33751,
  "Humenné": 32834,
  "Levice": 32735,
  "Bardejov": 32293,
  "Liptovský Mikuláš": 31000,
  "Lučenec": 27739,
  "Piešťany": 27336,
  "Ružomberok": 26558,
  "Topoľčany": 24785,
  "Trebišov": 24649,
  "Čadca": 23941,
  "Rimavská Sobota": 23751,
  "Dubnica nad Váhom": 23550,
  "Pezinok": 23034,
  "Dunajská Streda": 22684,
  "Partizánske": 22269,
  "Vranov nad Topľou": 22245,
  "Šaľa": 21689,
  "Hlohovec": 21301,
  "Brezno": 20889,
  "Snina": 20289,
  "Senica": 20289,
  "Nové Mesto nad Váhom": 20071,
  "Senec": 19963,
  "Rožňava": 19045,
  "Žiar nad Hronom": 18852,
  "Dolný Kubín": 18665,
  "Bánovce nad Bebravou": 17750,
  "Púchov": 17561,
  "Malacky": 17467,
  "Handlová": 16890,
  "Kežmarok": 16346,
  "Stará Ľubovňa": 16327,
  "Sereď": 15444,
  "Skalica": 15022,
  "Galanta": 14990,
  "Kysucké Nové Mesto": 14953,
  "Levoča": 14722,
  "Detva": 14545,
  "Šamorín": 13350,
  "Sabinov": 12700,
  "Stupava": 12108,
  "Revúca": 11949,
  "Veľký Krtíš": 11755,
  "Myjava": 11514,
  "Zlaté Moravce": 11375,
  "Bytča": 11363,
  "Moldava nad Bodvou": 11307,
  "Holíč": 11156,
  "Nová Dubnica": 11040,
  "Svidník": 10752,
  "Kolárovo": 10586,
  "Fiľakovo": 10568,
  "Stropkov": 10438,
  "Štúrovo": 10186,
  "Banská Štiavnica": 10004,
  "Šurany": 9659,
  "Tvrdošín": 9201,
  "Modra": 9046,
  "Veľké Kapušany": 8862,
  "Stará Turá": 8764,
  "Krompachy": 8684,
  "Veľký Meder": 8612,
  "Vráble": 8551,
  "Sečovce": 8542,
  "Svit": 7923,
  "Krupina": 7912,
  "Námestovo": 7784,
  "Vrútky": 7748,
  "Turzovka": 7515,
  "Kráľovský Chlmec": 7426,
  "Hriňová": 7415,
  "Hnúšťa": 7411,
  "Liptovský Hrádok": 7397,
  "Hurbanovo": 7397,
  "Trstená": 7360,
  "Nová Baňa": 7314,
  "Šahy": 7219,
  "Tornaľa": 7114,
  "Krásno nad Kysucou": 6799,
  "Želiezovce": 6746,
  "Spišská Belá": 6733,
  "Lipany": 6542,
  "Medzilaborce": 6531,
  "Veľký Šariš": 6426,
  "Nemšová": 6414,
  "Sobrance": 6346,
  "Turčianske Teplice": 6330,
  "Žarnovica": 6170,
  "Gelnica": 6047,
  "Vrbové": 5907,
  "Rajec": 5787,
  "Svätý Jur": 5724,
  "Dobšiná": 5685,
  "Poltár": 5584,
  "Gabčíkovo": 5473,
  "Ilava": 5464,
  "Kremnica": 5258,
  "Sládkovičovo": 5209,
  "Gbely": 5073,
  "Nesvady": 5049,
  "Bojnice": 4992,
  "Šaštín-Stráže": 4965,
  "Sliač": 4959,
  "Brezová pod Bradlom": 4768,
  "Medzev": 4482,
  "Turany": 4278,
  "Strážske": 4250,
  "Nováky": 4203,
  "Trenčianske Teplice": 4162,
  "Leopoldov": 4097,
  "Giraltovce": 4089,
  "Tisovec": 4087,
  "Vysoké Tatry": 4009,
  "Spišské Podhradie": 3985,
  "Hanušovce nad Topľou": 3782,
  "Čierna nad Tisou": 3619,
  "Tlmače": 3520,
  "Spišské Vlachy": 3459,
  "Jelšava": 3289,
  "Podolínec": 3155,
  "Rajecké Teplice": 3023,
  "Spišská Stará Ves": 2230,
  "Modrý Kameň": 1624,
  "Dudince": 1395
}


In [443]:
for key in POD_VLIVEM_STRUCTURED:
    placeOfOffense = POD_VLIVEM_STRUCTURED[key]["placeOfOffense"]
    POD_VLIVEM_STRUCTURED[key]["citySize"] = cities.get(placeOfOffense, "n/a")
    

In [444]:
import re

def average_measurements(measure_str: str) -> str:
    # Handle NaN or None
    if pd.isna(measure_str):
        return measure_str
    
    # If multiple values separated by semicolon

    parts = [p.strip() for p in str(measure_str).split(";")]
    
    values = []
    units = []
    
    for part in parts:
        match = re.match(r"([\d.]+)\s*([a-zA-Zµ/]+)", part)
        if not match:
            return measure_str  # if parsing fails, return original
        value, unit = match.groups()
        unit = unit.replace("L", "l").replace("liter", "l")
        values.append(float(value))
        units.append(unit)
    
    # Check if all units are identical
    if len(set(units)) == 1:
        avg = sum(values) / len(values)
        return f"{avg:.2f} {units[0]}"
    else:
        return measure_str
    
    
    # If not multiple values, just return original
    return measure_str



In [445]:
for key in POD_VLIVEM_STRUCTURED:
    amountUsed = POD_VLIVEM_STRUCTURED[key]["amountUsed"]
    substanceUsed = POD_VLIVEM_STRUCTURED[key]["substanceUsed"]
    if len(set(substanceUsed.split(";"))) == 1:
        amount_with_unit = average_measurements(amountUsed)
        #print(f"{substanceUsed}: {amount_with_unit}")
        parts = amount_with_unit.split(" ")

        if len(parts) == 2:
            try:
                amount = float(parts[0])
            except:
                print(f"Could not process {amountUsed}")
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = "n/a"      
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = "n/a"         
                parts_units = parts[1].split("/")
                
            if "alcohol" in substanceUsed and parts[0] == "ng/l":
                print("will divide")
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = amount/1000
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = "mg/l"         
            elif "alcohol" not in substanceUsed and parts[1] == "µg/ml":
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = amount * 1000
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = "ng/ml"         
            elif  "alcohol" in substanceUsed and parts[1] == "mg/l" or "alcohol" not in substanceUsed and parts[1] == "ng/ml":
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = amount
                POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = parts[1]
            elif parts[1] not in [ "ng/l",  "mg/l",  "ng/ml",  "mg/ml", "µg/ml" ]:
                print( parts[1])
        else:
            POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = "n/a"
            POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = "n/a"         
    else:
        POD_VLIVEM_STRUCTURED[key]["avgAmountUsed"] = "n/a"
        POD_VLIVEM_STRUCTURED[key]["avgAmountUsedUnit"] = "n/a"         

Mg/l
Could not process X.XX mg/l


In [448]:
import numpy as np
df_pod_vlivem = pd.DataFrame.from_dict(POD_VLIVEM_STRUCTURED, orient="index").reset_index()
df_pod_vlivem.replace("nan", np.nan, inplace=True)
df_pod_vlivem.replace("n/a", np.nan, inplace=True)
df_pod_vlivem["citySize"] = df_pod_vlivem["citySize"].astype("Int64")
df_pod_vlivem["substanceUsed"] = df_pod_vlivem["substanceUsed"].str.replace("amfetamínu","amfetamín").str.replace("metamfetamín","methamphetamine").str.replace("amfetamín","amphetamine")

In [449]:
order = ['index', 'substanceUsed', 'avgAmountUsed', 'avgAmountUsedUnit', 'amountUsed',
       'measuringMethod', 'measuringTime',  'measuringDevice', 'refusedTesting',
       'dateOfOffense', 'timeOfOffense', 'placeOfOffense', 'citySize', 'placeDetails', 'typeOfRoad',
       'vehicleModel', 'vehicleType', 'checkedBy', 
       'reasonForCheck', 'reasonForCheckCategory', 'licenseStatus', 'lawBroken', 'note', 'sentence'
       ]
df_pod_vlivem = df_pod_vlivem[order]

In [450]:
df_pod_vlivem.to_csv("pod_vlivem_attributes_test.csv", index=False)

In [391]:
df_pod_vlivem

,index,substanceUsed,avgAmountUsed,avgAmountUsedUnit,amountUsed,measuringMethod,measuringTime,measuringDevice,refusedTesting,dateOfOffense,...,typeOfRoad,vehicleModel,vehicleType,checkedBy,reasonForCheck,reasonForCheckCategory,licenseStatus,lawBroken,note,sentence
0,95214574,alcohol,0.58,mg/l,0.64 mg/l; 0.55 mg/l; 0.54 mg/l,breath; breath; breath,03:52; 04:24; 04:54,"AlcoQuant 6020 plus, výrobné číslo A409530",False,01.01.2022,...,class_II,VW Golf,car,hliadkou ODI OR PZ Veľký Krtíš,zastavená a kontrolovaná,traffic_check,valid,NaN,Vozidlo malo evidenčné číslo VK 823CM a smer j...,- dňa 01.01.2022 v čase o 03.50 h viedla pod v...
1,95255832,alcohol,0.74,mg/l,0.74 mg/l,breath,NaN,NaN,False,01.01.2022,...,urban,Seat Altea,car,Obvodné oddelenie Policajného zboru Zlaté Moravce,Kontrola vozidla a vodiča,traffic_check,valid,NaN,Vozidlo malo evidenčné číslo LV-XXX EK.,že : po predchádzajúcom požití väčšieho množs...
2,95309650,methamphetamine; amphetamine,NaN,NaN,198.05 ng/ml; 33.56 ng/ml,blood; blood,10:30; 10:30,NaN,False,13.05.2021,...,urban,Dacia Logan,car,policajnou hliadkou,javil známky požitia omamných a psychotropných...,suspected_intoxication,valid,NaN,Podľa znaleckého posudku z odvetvia psychiatri...,", ž e dňa 13. mája 2021 v čase okolo 09.50 ho..."
3,95331065,alcohol,0.26,mg/l,0.26 mg/l,breath,NaN,NaN,False,04.01.2022,...,class_I,anonymized,car,hliadkou ODI Stará Ľubovňa,NaN,traffic_check,banned,NaN,Zákaz činnosti viesť motorové vozidlá na dobu ...,dňa 04.01.2022 v čase okolo 10.11 hod. pod vpl...
4,114902217,cannabis,44.50,ng/ml,7.00 ng/ml; 82.00 ng/ml,blood; blood,23:45; 23:45,NaN,False,19.11.2021,...,class_II,Škoda Octavia,car,OO PZ Čadca,"Kontrolovaný hliadkou OO PZ Čadca, nakoľko jav...",suspected_intoxication,valid,NaN,Vodič sa podrobil dychovej skúške na zistenie ...,dňa 19.11.2021 okolo 21:50 h viedol osobné mot...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1770,101536112,alcohol,0.59,mg/l,0.59 mg/l,breath,03:51,NaN,False,03.04.2022,...,urban,Mitsubishi Carisma,car,policajnou hliadkou,na zistenie požitia alkoholu,suspected_intoxication,valid,§ 4 odsek 2 písmeno d) zákona č. 8/2009 Z.z. o...,Evidenčné číslo vozidla bolo SE-XXXBE a jeho m...,že : dňa 03.04.2022 v čase o 03.51 hod. na ul...
1771,106473883,alcohol,0.61,mg/l,0.54 mg/l; 0.66 mg/l; 0.62 mg/l,breath; breath; breath,16:45; 17:06; 17:25,AlcoQuant 6020 plus,False,11.06.2022,...,NaN,Babeta,motorcycle,Okresný dopravný inšpektorát Trnava,jazda na motocykli bez ochrannej prilby a zo s...,traffic_violation; suspected_intoxication,valid,NaN,Sériové číslo prístroja AlcoQuant 6020 plus bo...,", ž e dňa 11.06.2022 v čase o 16:35 hodine vi..."
1772,108365243,alcohol,0.88,mg/l,0.89 mg/l; 0.82 mg/l; 0.95 mg/l; 0.86 mg/l,breath; breath; breath; breath,21:02; 21:17; 21:36; 22:06,AlcoQuant 6020 plus,False,18.06.2022,...,NaN,Mercedes Benz Vito,van,hliadkou OO PZ Partizánske,predpísaným spôsobom zastavený a kontrolovaný,traffic_check,valid,NaN,Vodič nemal pri sebe doklady potrebné na veden...,: dňa 18.06.2022 v čase o 20:50 h viedol osob...
1773,113761793,alcohol,1.22,mg/l,1.22 mg/l,breath,NaN,NaN,False,19.09.2022,...,urban,anonymized,car,OVS ODI OR PZ Košice,prekročenie povolenej rýchlosti,speeding,valid,NaN,Okrem extrahovaných údajov text uvádza hodnotu...,v Košiciach po ul. Gemerskej k ul. Rastislavov...


In [114]:
non_na_series = list(df_pod_vlivem["reasonForCheck"].dropna())

In [125]:
nan_info = df_pod_vlivem.isna().sum().to_frame("NaN count")
nan_info["NaN %"] = (df_pod_vlivem.isna().mean() * 100).round(2)
print(nan_info["NaN %"])

index                 0.00
substanceUsed         4.08
substanceUsedType     8.67
amountUsed           12.76
measuringMethod       8.16
measuringTime        38.27
avgAmountUsed        12.76
measuringDevice      41.84
refusedTesting        0.00
dateOfOffense         0.51
timeOfOffense         0.00
placeOfOffense        1.53
citySize             57.14
placeDetails          0.51
typeOfRoad           17.35
vehicleModel          1.02
vehicleType           0.00
checkedBy            11.73
reasonForCheck       69.39
licenseStatus         0.00
lawBroken            84.18
note                  0.00
sentence              0.00
Name: NaN %, dtype: float64


In [10]:
final = pd.read_csv("pod_vlivem_attributes_1775.csv")

In [404]:
errors = 0
for n,item in enumerate(list(final["measuringMethod"])):
    items = str(item).split(";")
    if any([item.strip() not in ["breath", "blood", "urine", "nan", "na"] for item in items]):
        errors += 1
        print(f"{n}:{item}")


131:breath; breath; breath; blood; other
143:dych; dych
955:n/a; n/a
1225:moč; moč
1350:krv; krv
1537:blood; other; other
1640:blood; other
1762:breath; blood; other


## Validations

In [ ]:
from dateutil.parser import parse

def is_date_after_2000(string):
    try:
        dt = parse(string, fuzzy=False)
        return dt.year > 2000
    except ValueError:
        return False

def check_date_after_2000(row, n, fmt="%H:%M"):
    string = str(row["dateOfOffense"])
    if time == "nan":
        return True
    try:
        dt = parse(string, fuzzy=False)
        if dt.year > 2000:
            return True
        else:
            return False
    except ValueError:
        print(f"dateOfOffense value '{string}' does not match the requirements.")
        return False

errors = 0
for n,item in enumerate(list(final["dateOfOffense"])):
    try:
        if not is_date_after_2000( item):
            errors += 1
            print(f"{n}:{item}")
    except:
        errors+=1

In [ ]:
from datetime import datetime
import math

def is_time(string, fmt="%H:%M"):
    try:
        datetime.strptime(string, fmt)
        return True
    except ValueError:
        return False

def check_time(row, n, fmt="%H:%M"):
    time = str(row["timeOfOffense"])
    if time == "nan":
        return True
    try:
        datetime.strptime(time, fmt)
        return True
    except ValueError:
        print(f"timeOfOffense value '{time}' does not match the requirements.")
        return False

errors = 0
for n,item in enumerate(list(final["timeOfOffense"])):
    try:
        if not is_time( item):
            errors += 1
            print(f"{n}:{item}")
    except:
        errors+=1


In [ ]:
def check_substance(row, n):   

    substances = [
    "alcohol",
    "cannabis",
    "methamphetamine",
    "amphetamine",
    "cocaine",
    "mdma",
    "hashish",
    "morphine",
    "barbiturate",
    "benzodiazepine",
    "methadon",
    "heroin",
    "other",
    "nan"
    ]
    field = row["substanceUsed"]
    items = str(field).split(";")
    if any([item.strip() not in substances for item in items]):
        print(f"{n}:{field}")
        print(f"substanceUsed value '{item}' does not match the requirements.")
        return False
    else:
        #print(f"{n}:{items} is ok")
        return True

errors = 0
for n,row in final.iterrows():
    check_substance(row, n)
